# 01 — MERRA-2 Climate Data Exploration

**Dataset:** NASA MERRA-2 monthly reanalysis, 1995–2024, India state-level aggregation  
**Task:** Climate condition classification (Normal / Drought / Wet_Flood / Heat_Extreme / Cold_Extreme)  

This notebook explores the processed dataset before any modeling.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml

from src.data.label import LABEL_MAP, LABEL_NAMES

# Publication style
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

with open('../config/config.yaml') as f:
    config = yaml.safe_load(f)

CLASS_COLORS = {
    'Normal': '#4CAF50',
    'Drought': '#FF9800',
    'Wet_Flood': '#2196F3',
    'Heat_Extreme': '#F44336',
    'Cold_Extreme': '#9C27B0',
}
print('Imports OK')

In [ ]:
# Load labeled dataset
df = pd.read_csv('../data/processed/merra2_india_labeled.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Dataset overview
print(f'Years: {df.year.min()} – {df.year.max()}')
print(f'States: {df.state.nunique()} unique states/UTs')
print(f'Months: {df.month.nunique()} calendar months')
print(f'\nMissing values:')
print(df.isnull().sum())
print(f'\nDescriptive statistics:')
df.describe()

## 1. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label_name'].value_counts()
colors = [CLASS_COLORS[n] for n in counts.index]

# Bar chart
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Climate Condition')
axes[0].set_ylabel('Sample Count')
axes[0].set_title('Class Distribution (Absolute)')
axes[0].tick_params(axis='x', rotation=30)
for i, (name, count) in enumerate(counts.items()):
    axes[0].text(i, count + 50, f'{count:,}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Distribution (%)')

plt.tight_layout()
plt.savefig('../results/figures/class_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

## 2. Plotly Interactive Map — Mean Temperature by State

In [ ]:
# Load shapefile for map
gdf = gpd.read_file('../data/shapefiles/india_states.geojson')

# Normalize state name column
name_col = [c for c in gdf.columns if any(k in c.lower() for k in ['state', 'name', 'st_nm'])][0]
gdf = gdf.rename(columns={name_col: 'state'})
geojson = json.loads(gdf.to_json())

# Mean T2M per state (convert K to °C)
state_temp = df.groupby('state')['T2M'].mean().reset_index()
state_temp['T2M_C'] = state_temp['T2M'] - 273.15

fig = px.choropleth(
    state_temp,
    geojson=geojson,
    locations='state',
    featureidkey='properties.state',
    color='T2M_C',
    color_continuous_scale='RdYlBu_r',
    range_color=[15, 35],
    title='Mean 2m Temperature by Indian State (1995–2024, °C)',
    labels={'T2M_C': 'Temp (°C)'},
)
fig.update_geos(
    fitbounds='locations',
    visible=False,
    showcoastlines=True,
    coastlinecolor='gray',
)
fig.update_layout(
    margin=dict(l=0, r=0, t=40, b=0),
    height=550,
)
fig.show()

## 3. Animated Map — Annual Mean Temperature Over Time

In [ ]:
annual_state = df.groupby(['year', 'state'])['T2M'].mean().reset_index()
annual_state['T2M_C'] = annual_state['T2M'] - 273.15

fig = px.choropleth(
    annual_state,
    geojson=geojson,
    locations='state',
    featureidkey='properties.state',
    color='T2M_C',
    animation_frame='year',
    color_continuous_scale='RdYlBu_r',
    range_color=[15, 35],
    title='Annual Mean Temperature by State (animated)',
    labels={'T2M_C': 'Temp (°C)'},
)
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(height=550, margin=dict(l=0, r=0, t=40, b=0))
fig.show()

## 4. Time Series — Key States

In [ ]:
key_states = ['Delhi', 'Kerala', 'Rajasthan', 'West Bengal', 'Himachal Pradesh']
key_states = [s for s in key_states if s in df['state'].unique()]

df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
df_key = df[df['state'].isin(key_states)].copy()
df_key['T2M_C'] = df_key['T2M'] - 273.15
df_key['PRECTOT_mm'] = df_key['PRECTOT'] * 86400 * 30  # kg/m2/s -> mm/month

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['2m Temperature (°C)', 'Precipitation (mm/month)'])

for state in key_states:
    s = df_key[df_key['state'] == state].sort_values('date')
    fig.add_trace(go.Scatter(x=s['date'], y=s['T2M_C'], name=state,
                              mode='lines', line=dict(width=1.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=s['date'], y=s['PRECTOT_mm'], name=state,
                              mode='lines', line=dict(width=1.5),
                              showlegend=False), row=2, col=1)

fig.update_layout(
    title='Climate Variables for Selected Indian States (1995–2024)',
    height=600,
    legend=dict(orientation='h', y=-0.1),
)
fig.show()

## 5. Correlation Heatmap

In [ ]:
feature_cols = ['T2M', 'PRECTOT', 'QV2M', 'U10M', 'V10M', 'PS', 'SLP',
                'SWGDN', 'LWGNT', 'CLDTOT', 'EVAP']
feature_cols = [c for c in feature_cols if c in df.columns]

corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix (MERRA-2 Variables)')
plt.tight_layout()
plt.savefig('../results/figures/correlation_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Box Plots — Variables by Climate Condition

In [ ]:
df_plot = df.copy()
df_plot['T2M_C'] = df_plot['T2M'] - 273.15
df_plot['PRECTOT_mm'] = df_plot['PRECTOT'] * 86400 * 30

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
order = ['Normal', 'Drought', 'Wet_Flood', 'Heat_Extreme', 'Cold_Extreme']
order = [o for o in order if o in df_plot['label_name'].unique()]
palette = {k: v for k, v in CLASS_COLORS.items() if k in order}

sns.boxplot(data=df_plot, x='label_name', y='T2M_C', order=order,
            palette=palette, ax=axes[0], linewidth=0.8)
axes[0].set_xlabel('Climate Condition')
axes[0].set_ylabel('Temperature (°C)')
axes[0].set_title('Temperature Distribution by Class')
axes[0].tick_params(axis='x', rotation=25)

sns.boxplot(data=df_plot, x='label_name', y='PRECTOT_mm', order=order,
            palette=palette, ax=axes[1], linewidth=0.8)
axes[1].set_xlabel('Climate Condition')
axes[1].set_ylabel('Precipitation (mm/month)')
axes[1].set_title('Precipitation Distribution by Class')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig('../results/figures/boxplots_by_class.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Spatial Distribution of Climate Conditions

In [ ]:
# Most common label per state
dominant_class = df.groupby('state')['label_name'].agg(
    lambda x: x.value_counts().index[0]
).reset_index()
dominant_class.columns = ['state', 'dominant_condition']

fig = px.choropleth(
    dominant_class,
    geojson=geojson,
    locations='state',
    featureidkey='properties.state',
    color='dominant_condition',
    color_discrete_map=CLASS_COLORS,
    title='Dominant Climate Condition by State (1995–2024)',
    labels={'dominant_condition': 'Condition'},
)
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(height=550, margin=dict(l=0, r=0, t=40, b=0))
fig.show()

In [ ]:
# Fraction of each condition per state (stacked bar)
state_cond = df.groupby(['state', 'label_name']).size().unstack(fill_value=0)
state_cond_pct = state_cond.div(state_cond.sum(axis=1), axis=0) * 100
state_cond_pct = state_cond_pct.sort_values('Heat_Extreme', ascending=False)

fig = go.Figure()
for cond in ['Heat_Extreme', 'Cold_Extreme', 'Drought', 'Wet_Flood', 'Normal']:
    if cond in state_cond_pct.columns:
        fig.add_trace(go.Bar(
            name=cond,
            x=state_cond_pct.index,
            y=state_cond_pct[cond],
            marker_color=CLASS_COLORS[cond],
        ))

fig.update_layout(
    barmode='stack',
    title='Climate Condition Frequency (%) by State',
    xaxis_tickangle=-45,
    yaxis_title='Percentage of months (%)',
    height=500,
    legend=dict(orientation='h', y=1.02),
)
fig.show()

## 8. Seasonal Patterns

In [ ]:
monthly_counts = df.groupby(['month', 'label_name']).size().unstack(fill_value=0)
monthly_counts_pct = monthly_counts.div(monthly_counts.sum(axis=1), axis=0) * 100

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig = go.Figure()
for cond in ['Heat_Extreme', 'Cold_Extreme', 'Drought', 'Wet_Flood', 'Normal']:
    if cond in monthly_counts_pct.columns:
        fig.add_trace(go.Scatter(
            x=month_names,
            y=monthly_counts_pct[cond],
            name=cond,
            mode='lines+markers',
            line=dict(color=CLASS_COLORS[cond], width=2),
            marker=dict(size=6),
        ))

fig.update_layout(
    title='Seasonal Distribution of Climate Conditions (all states, all years)',
    xaxis_title='Month',
    yaxis_title='% of state-month records',
    height=450,
    legend=dict(orientation='h', y=-0.2),
)
fig.show()

## Summary

Key observations from the EDA:
- Dataset covers **{df.year.min()}–{df.year.max()}** across **{df.state.nunique()} Indian states/UTs**
- Strong seasonality: monsoon months (Jun–Sep) dominate Wet_Flood events; pre-monsoon (Apr–Jun) has Heat Extremes
- Rajasthan and Gujarat show the highest drought frequency; northeastern states have the most Wet_Flood events
- T2M and PRECTOT are moderately anti-correlated, supporting their combined use for classification

**Next:** Phase 2 — Classical ML models (Random Forest, SVM, XGBoost, Neural Network)